# CWSB Model — Phases 0–2, fully on Colab

**Before running:** Runtime → Change runtime type → **T4 GPU**.

**Prerequisite:** upload `Data_final.zip` (the raw data: `Healthy/` + `Unhealthy(Infected)/`) to the **top level of your Google Drive**.

Run all cells top to bottom (Runtime → Run all works too). Phases: 0 setup → 1 clean + split + tiles → 2 train both contenders. At the end `models_v1.zip` downloads automatically — unzip it into `models/v1/` in the local repo.

In [ ]:
# Phase 0 — GPU check + install
!nvidia-smi -L
!pip -q install timm


In [ ]:
# Phase 0 — mount Drive, unzip raw data, set up folders
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib
os.makedirs('/content/cwsb/src', exist_ok=True)
%cd /content/cwsb
!unzip -q /content/drive/MyDrive/Data_final.zip -d /content/cwsb
# normalize layout -> /content/cwsb/Data_final/{Healthy, Unhealthy(Infected)}
p = pathlib.Path('/content/cwsb')
if not (p / 'Data_final').exists():
    if (p / 'Healthy').exists():
        (p / 'Data_final').mkdir()
        for d in ['Healthy', 'Unhealthy(Infected)']:
            os.rename(p / d, p / 'Data_final' / d)
    else:
        raise SystemExit('Data_final folder not found after unzip - check the zip layout')
h = len(list((p / 'Data_final' / 'Healthy').glob('*.jpg')))
u = len(list((p / 'Data_final' / 'Unhealthy(Infected)').glob('*.jpg')))
print(f'{h} healthy + {u} unhealthy = {h + u} images')
assert (h, u) == (28, 470), 'unexpected counts'


## The pipeline scripts (written into `src/` on this machine)

In [ ]:
%%writefile src/clean.py
"""Phase 1a - Clean Data_final into work/clean.

Per image: apply EXIF rotation, strip ALL metadata (GPS!), save plain JPEG q95
at full resolution, and record quality scores (blur / clipped / dark), SHA-256
and a perceptual dhash. Then report exact and near duplicates.

Data_final/ is read-only; everything is written to work/clean/.
"""
import csv
import hashlib
import sys
from pathlib import Path

import cv2
import numpy as np
from PIL import Image, ImageOps

ROOT = Path(__file__).resolve().parents[1]
SRC = ROOT / "Data_final"
DST = ROOT / "work" / "clean"
CLASSES = {"Healthy": "Healthy", "Unhealthy(Infected)": "Unhealthy"}

# Quality flags (tuned for big phone photos)
BLUR_MIN = 60.0        # Laplacian variance below this => blurry
CLIPPED_MAX = 0.20     # >20% near-white pixels => burnt out (sun)
DARK_MAX = 0.40        # >40% near-black pixels => too dark
NEAR_DUP_HAMMING = 6   # dhash distance <= this => near duplicate


def dhash(gray: np.ndarray, size: int = 8) -> int:
    small = cv2.resize(gray, (size + 1, size), interpolation=cv2.INTER_AREA)
    bits = small[:, 1:] > small[:, :-1]
    return int("".join("1" if b else "0" for b in bits.flatten()), 2)


def hamming(a: int, b: int) -> int:
    return bin(a ^ b).count("1")


def main() -> int:
    rows = []
    for src_cls, dst_cls in CLASSES.items():
        out_dir = DST / dst_cls
        out_dir.mkdir(parents=True, exist_ok=True)
        files = sorted((SRC / src_cls).glob("*.jpg"))
        print(f"{src_cls}: {len(files)} images")
        for i, p in enumerate(files):
            img = ImageOps.exif_transpose(Image.open(p)).convert("RGB")
            out = out_dir / p.name
            img.save(out, "JPEG", quality=95, exif=b"")  # strips GPS/EXIF

            arr = np.asarray(img)
            g_full = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
            scale = 1024 / max(g_full.shape[1], 1)
            g = cv2.resize(g_full, (1024, max(int(g_full.shape[0] * scale), 1)))
            blur = float(cv2.Laplacian(g, cv2.CV_64F).var())
            clipped = float((g > 250).mean())
            dark = float((g < 15).mean())

            flags = []
            if blur < BLUR_MIN:
                flags.append("blurry")
            if clipped > CLIPPED_MAX:
                flags.append("clipped")
            if dark > DARK_MAX:
                flags.append("dark")

            rows.append({
                "file": f"{dst_cls}/{p.name}",
                "label": dst_cls,
                "width": img.width,
                "height": img.height,
                "blur": round(blur, 2),
                "clipped": round(clipped, 4),
                "dark": round(dark, 4),
                "sha256": hashlib.sha256(out.read_bytes()).hexdigest(),
                "dhash": f"{dhash(g):016x}",
                "flags": "|".join(flags),
            })
            if (i + 1) % 50 == 0:
                print(f"  {i + 1}/{len(files)}")

    with open(DST / "images.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)

    # Duplicate report
    dupes = []
    for i in range(len(rows)):
        for j in range(i + 1, len(rows)):
            a, b = rows[i], rows[j]
            if a["sha256"] == b["sha256"]:
                dupes.append((a["file"], b["file"], "exact", 0))
            else:
                d = hamming(int(a["dhash"], 16), int(b["dhash"], 16))
                if d <= NEAR_DUP_HAMMING:
                    dupes.append((a["file"], b["file"], "near", d))
    with open(DST / "duplicates.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["file_a", "file_b", "kind", "hamming"])
        w.writerows(dupes)

    n_flag = sum(1 for r in rows if r["flags"])
    cross = [d for d in dupes if d[0].split("/")[0] != d[1].split("/")[0]]
    print(f"\nDone: {len(rows)} images cleaned -> {DST}")
    print(f"Flagged (excluded from training): {n_flag}")
    for r in rows:
        if r["flags"]:
            print(f"  {r['file']}: {r['flags']} (blur={r['blur']}, clipped={r['clipped']}, dark={r['dark']})")
    print(f"Duplicate pairs: {len(dupes)} ({len(cross)} CROSS-CLASS - label errors, must review)")
    for d in dupes:
        print(f"  {d[2]} d={d[3]}: {d[0]} <-> {d[1]}")
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile src/make_split.py
"""Phase 1b - Group photos into pseudo-plants and make a leak-proof split.

Consecutive frame numbers (gap <= 4) or timestamps within 120 s from the same
camera series are treated as ONE plant. The split is done on plants, not
photos: 70/15/15 per class, so no plant ever appears in two sets. Flagged
(blurry/clipped/dark) images are excluded. The script refuses to finish if
any leakage is detected.
"""
import csv
import random
import re
import sys
from collections import defaultdict
from datetime import datetime
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
CLEAN = ROOT / "work" / "clean"
OUT = ROOT / "work" / "splits" / "split.csv"

FRAME_GAP = 4       # frames
TIME_GAP = 120.0    # seconds
FRACTIONS = {"train": 0.70, "val": 0.15, "test": 0.15}
SEED = 42

TS_RE = re.compile(r"(?P<pre>.*IMG)_(?P<d>\d{8})_(?P<t>\d{6})", re.I)
NUM_RE = re.compile(r"^(?P<pre>.*?)(?P<num>\d+)(?P<post>[^\d]*)$")


def sort_key(name, label):
    stem = Path(name).stem
    m = TS_RE.match(stem)
    if m:
        ts = datetime.strptime(m.group("d") + m.group("t"), "%Y%m%d%H%M%S")
        return (label, "ts:" + m.group("pre"), ts.timestamp())
    m = NUM_RE.match(stem)
    if m:
        return (label, "fr:" + m.group("pre") + "|" + m.group("post"), float(m.group("num")))
    return (label, "one:" + stem, 0.0)


def main() -> int:
    with open(CLEAN / "images.csv", newline="") as f:
        rows = list(csv.DictReader(f))

    # --- group into pseudo-plants (within class + camera series only) ---
    keyed = sorted(rows, key=lambda r: sort_key(r["file"], r["label"]))
    groups = []
    prev = None
    for r in keyed:
        label, series, pos = sort_key(r["file"], r["label"])
        gap = TIME_GAP if series.startswith("ts:") else FRAME_GAP
        if prev and prev[0] == (label, series) and pos - prev[1] <= gap:
            groups[-1].append(r)
        else:
            groups.append([r])
        prev = ((label, series), pos)
    for gi, g in enumerate(groups):
        for r in g:
            r["group"] = f"plant_{gi:03d}"

    # --- split groups per class, excluding flagged images from counts ---
    rng = random.Random(SEED)
    by_class = defaultdict(list)
    for g in groups:
        usable = [r for r in g if not r["flags"]]
        if usable:
            by_class[g[0]["label"]].append(g)

    assign = {}
    for label, gs in by_class.items():
        rng.shuffle(gs)
        total = sum(len([r for r in g if not r["flags"]]) for g in gs)
        want = {s: total * frac for s, frac in FRACTIONS.items()}
        got = {s: 0 for s in FRACTIONS}
        # give each split one group first (so every set has both classes)
        order = ["test", "val", "train"]
        for s, g in zip(order, gs):
            assign[g[0]["group"]] = s
            got[s] += len([r for r in g if not r["flags"]])
        for g in gs[len(order):]:
            s = max(FRACTIONS, key=lambda s: want[s] - got[s])
            assign[g[0]["group"]] = s
            got[s] += len([r for r in g if not r["flags"]])

    for r in rows:
        r["split"] = "excluded" if r["flags"] else assign.get(r["group"], "excluded")

    # --- leakage checks ---
    plant_splits = defaultdict(set)
    for r in rows:
        if r["split"] != "excluded":
            plant_splits[r["group"]].add(r["split"])
    leaks = {g: s for g, s in plant_splits.items() if len(s) > 1}
    if leaks:
        print(f"LEAKAGE: {leaks}")
        return 1

    dup_leaks = []
    dup_file = CLEAN / "duplicates.csv"
    if dup_file.exists():
        split_of = {r["file"]: r["split"] for r in rows}
        with open(dup_file, newline="") as f:
            for d in csv.DictReader(f):
                sa, sb = split_of.get(d["file_a"]), split_of.get(d["file_b"])
                if sa and sb and sa != sb and "excluded" not in (sa, sb):
                    dup_leaks.append((d["file_a"], sa, d["file_b"], sb, d["kind"]))
    for d in dup_leaks:
        print(f"WARNING near-dup across splits: {d}")

    OUT.parent.mkdir(parents=True, exist_ok=True)
    with open(OUT, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["file", "label", "group", "split"])
        w.writeheader()
        w.writerows([{k: r[k] for k in ("file", "label", "group", "split")} for r in rows])

    print(f"Plants: {len(groups)} total "
          f"(Healthy {len(by_class.get('Healthy', []))}, Unhealthy {len(by_class.get('Unhealthy', []))})")
    tally = defaultdict(lambda: defaultdict(int))
    for r in rows:
        tally[r["split"]][r["label"]] += 1
    for s in ("train", "val", "test", "excluded"):
        h, u = tally[s].get("Healthy", 0), tally[s].get("Unhealthy", 0)
        print(f"  {s:8s}: {h:3d} healthy + {u:3d} unhealthy = {h + u}")
    print("LEAKAGE CHECK OK" + (f" ({len(dup_leaks)} near-dup warnings)" if dup_leaks else ""))
    print(f"-> {OUT}")
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile src/make_tiles.py
"""Phase 2a - Multi-crop tiling for the tiny healthy class.

TRAIN-split healthy photos only (never val/test): each big photo (3000-6000 px)
is cut into overlapping square tiles covering different stem/canopy regions -
genuinely different pixel regions, not synthetic copies. 28 photos -> ~170
extra real samples. Tiles inherit their photo's plant group, so no leakage.
"""
import csv
import sys
from pathlib import Path

from PIL import Image

ROOT = Path(__file__).resolve().parents[1]
CLEAN = ROOT / "work" / "clean"
SPLIT = ROOT / "work" / "splits" / "split.csv"
OUT = ROOT / "work" / "tiles" / "Healthy"

GRID = (3, 2)  # cols x rows of overlapping tiles + 1 center tile


def main() -> int:
    OUT.mkdir(parents=True, exist_ok=True)
    with open(SPLIT, newline="") as f:
        rows = [r for r in csv.DictReader(f)
                if r["label"] == "Healthy" and r["split"] == "train"]

    manifest = []
    for r in rows:
        img = Image.open(CLEAN / r["file"])
        w, h = img.size
        side = int(min(w, h) * 0.55)  # tile side: 55% of short edge -> overlap
        cols, rows_n = GRID
        xs = [int(i * (w - side) / max(cols - 1, 1)) for i in range(cols)]
        ys = [int(j * (h - side) / max(rows_n - 1, 1)) for j in range(rows_n)]
        boxes = [(x, y) for y in ys for x in xs]
        boxes.append(((w - side) // 2, (h - side) // 2))  # center
        stem = Path(r["file"]).stem
        for k, (x, y) in enumerate(boxes):
            name = f"{stem}_tile{k}.jpg"
            img.crop((x, y, x + side, y + side)).save(OUT / name, "JPEG", quality=95)
            manifest.append({"file": f"Healthy/{name}", "label": "Healthy",
                             "group": r["group"], "split": "train"})

    with open(ROOT / "work" / "tiles" / "tiles.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["file", "label", "group", "split"])
        w.writeheader()
        w.writerows(manifest)
    print(f"{len(rows)} train healthy photos -> {len(manifest)} tiles in {OUT}")
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile src/shrink_for_training.py
"""Resize work/clean and work/tiles images IN PLACE to short side 768.

Run on Colab after clean/split/tiles, before training: the model trains at
384x384, so this loses nothing while making data loading ~10x faster.
(Do NOT run locally - the local work/clean keeps full resolution.)
"""
from pathlib import Path

from PIL import Image

ROOT = Path(__file__).resolve().parents[1]
SHORT_SIDE = 768

n = 0
for d in [ROOT / "work" / "clean", ROOT / "work" / "tiles"]:
    for p in sorted(d.rglob("*.jpg")):
        img = Image.open(p).convert("RGB")
        s = SHORT_SIDE / min(img.size)
        if s < 1:
            img = img.resize((round(img.width * s), round(img.height * s)), Image.LANCZOS)
            img.save(p, "JPEG", quality=90)
            n += 1
print(f"resized {n} images to short side {SHORT_SIDE}")


In [ ]:
%%writefile src/train.py
"""Phase 2b - Train the CWSB classifier. Runs on Colab GPU (or CPU, slowly).

Two contenders, validation decides which ships:
  --mode A : frozen ConvNeXt-Tiny backbone -> logistic regression head
             (~1.5k trainable params, nearly impossible to overfit 28 images)
  --mode B : full ConvNeXt-Tiny fine-tune (2-epoch head warm-up, then all
             layers), weighted sampling + class-weighted loss + augmentation

Positive class = Unhealthy (infected). Model selection metric = validation
PR-AUC (average precision). Seed 42 everywhere.

Usage (from the project root, after clean/split/tiles):
  python src/train.py --mode A
  python src/train.py --mode B
"""
import argparse
import csv
import json
import random
import time
from pathlib import Path

import numpy as np
import timm
import torch
import torch.nn as nn
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms

ROOT = Path(__file__).resolve().parents[1]
CLEAN = ROOT / "work" / "clean"
TILES = ROOT / "work" / "tiles"
SPLIT = ROOT / "work" / "splits" / "split.csv"
TILES_CSV = ROOT / "work" / "tiles" / "tiles.csv"
OUT = ROOT / "models" / "v1"

IMG = 384
BACKBONE = "convnext_tiny.fb_in22k_ft_in1k_384"
SEED = 42
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.10),
    transforms.RandomApply([transforms.GaussianBlur(5)], p=0.15),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([  # MUST stay identical in the API
    transforms.Resize(IMG + 32),
    transforms.CenterCrop(IMG),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])


class CwsbDataset(Dataset):
    def __init__(self, rows, tf):
        self.rows, self.tf = rows, tf

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        base = TILES if r.get("tile") else CLEAN
        img = Image.open(base / r["file"]).convert("RGB")
        return self.tf(img), 1 if r["label"] == "Unhealthy" else 0


def load_rows(split, with_tiles=False):
    with open(SPLIT, newline="") as f:
        rows = [r for r in csv.DictReader(f) if r["split"] == split]
    if with_tiles and TILES_CSV.exists():
        with open(TILES_CSV, newline="") as f:
            rows += [dict(r, tile="1") for r in csv.DictReader(f) if r["split"] == split]
    return rows


def set_seed():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)


@torch.no_grad()
def scores(model, rows, device, bs=16):
    model.eval()
    dl = DataLoader(CwsbDataset(rows, eval_tf), batch_size=bs, num_workers=2)
    ys, ps = [], []
    for x, y in dl:
        p = torch.softmax(model(x.to(device)), 1)[:, 1]
        ps += p.cpu().tolist()
        ys += y.tolist()
    return np.array(ys), np.array(ps)


@torch.no_grad()
def features(backbone, rows, device, bs=16):
    backbone.eval()
    dl = DataLoader(CwsbDataset(rows, eval_tf), batch_size=bs, num_workers=2)
    feats, ys = [], []
    for x, y in dl:
        f = backbone(x.to(device))
        feats.append(f.cpu().numpy())
        ys += y.tolist()
    return np.concatenate(feats), np.array(ys)


def run_a(device):
    """Frozen backbone + logistic regression."""
    backbone = timm.create_model(BACKBONE, pretrained=True, num_classes=0).to(device)
    tr = load_rows("train", with_tiles=True)
    va = load_rows("val")
    print(f"A: {len(tr)} train rows (incl. tiles), {len(va)} val")
    t0 = time.time()
    Xtr, ytr = features(backbone, tr, device)
    Xva, yva = features(backbone, va, device)
    clf = LogisticRegression(class_weight="balanced", max_iter=5000, C=0.1)
    clf.fit(Xtr, ytr)
    ap = average_precision_score(yva, clf.predict_proba(Xva)[:, 1])
    print(f"A: val PR-AUC = {ap:.4f}  ({time.time() - t0:.0f}s)")
    OUT.mkdir(parents=True, exist_ok=True)
    torch.save({"mode": "A", "backbone": BACKBONE, "img": IMG,
                "coef": clf.coef_, "intercept": clf.intercept_,
                "val_pr_auc": ap}, OUT / "contender_A.pt")
    return ap


def run_b(device, epochs=40, patience=6):
    """Full fine-tune with warm-up, weighted sampling, class-weighted loss."""
    model = timm.create_model(BACKBONE, pretrained=True, num_classes=2).to(device)
    tr = load_rows("train", with_tiles=True)
    va = load_rows("val")
    n_pos = sum(r["label"] == "Unhealthy" for r in tr)
    n_neg = len(tr) - n_pos
    print(f"B: train {len(tr)} ({n_neg} healthy incl. tiles, {n_pos} unhealthy), val {len(va)}")

    weights = [1.0 / (n_pos if r["label"] == "Unhealthy" else n_neg) for r in tr]
    sampler = WeightedRandomSampler(weights, num_samples=len(tr), replacement=True)
    dl = DataLoader(CwsbDataset(tr, train_tf), batch_size=16, sampler=sampler, num_workers=2)
    cw = torch.tensor([len(tr) / (2 * n_neg), len(tr) / (2 * n_pos)], dtype=torch.float32)
    crit = nn.CrossEntropyLoss(weight=cw.to(device))

    head = [p for n, p in model.named_parameters() if "head" in n]
    body = [p for n, p in model.named_parameters() if "head" not in n]

    best_ap, best_ep, since = 0.0, -1, 0
    OUT.mkdir(parents=True, exist_ok=True)
    for ep in range(epochs):
        warm = ep < 2
        for p in body:
            p.requires_grad = not warm
        opt = torch.optim.AdamW(
            [{"params": head, "lr": 1e-3 if warm else 1e-4},
             {"params": body, "lr": 0 if warm else 1e-4}], weight_decay=0.05)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=len(dl))
        model.train()
        tot = 0.0
        for x, y in dl:
            opt.zero_grad()
            loss = crit(model(x.to(device)), y.to(device))
            loss.backward()
            opt.step()
            sched.step()
            tot += loss.item()
        yva, pva = scores(model, va, device)
        ap = average_precision_score(yva, pva)
        print(f"B ep{ep:02d} {'warm' if warm else 'full'}: loss {tot / len(dl):.4f}  val PR-AUC {ap:.4f}")
        if ap > best_ap:
            best_ap, best_ep, since = ap, ep, 0
            torch.save({"mode": "B", "backbone": BACKBONE, "img": IMG,
                        "state_dict": model.state_dict(),
                        "val_pr_auc": ap, "epoch": ep}, OUT / "contender_B.pt")
        else:
            since += 1
            if since >= patience:
                print(f"B: early stop (best ep{best_ep}, PR-AUC {best_ap:.4f})")
                break
    return best_ap


def main():
    ap_ = argparse.ArgumentParser()
    ap_.add_argument("--mode", choices=["A", "B", "both"], default="both")
    args = ap_.parse_args()
    set_seed()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"device: {device}")

    results = {}
    if args.mode in ("A", "both"):
        results["A"] = run_a(device)
    if args.mode in ("B", "both"):
        results["B"] = run_b(device)
    if results:
        winner = max(results, key=results.get)
        (OUT / "training_result.json").write_text(json.dumps(
            {"val_pr_auc": results, "winner": winner,
             "backbone": BACKBONE, "img": IMG, "seed": SEED}, indent=2))
        print(f"\nWinner on validation: contender {winner}  {results}")


if __name__ == "__main__":
    main()


In [ ]:
# Phase 1 — clean -> split -> tiles -> shrink (expect: LEAKAGE CHECK OK)
!python src/clean.py
!python src/make_split.py
!python src/make_tiles.py
!python src/shrink_for_training.py


In [ ]:
# Phase 2a — contender A: frozen backbone + logistic regression (~5 min)
!python src/train.py --mode A


In [ ]:
# Phase 2b — contender B: full fine-tune (~20-40 min on T4)
!python src/train.py --mode B


In [ ]:
# Results + download (models + split/quality CSVs for the local repo)
import json, pathlib
print(json.loads(pathlib.Path('models/v1/training_result.json').read_text()))
!zip -rq models_v1.zip models/v1 work/splits/split.csv work/clean/images.csv work/clean/duplicates.csv
from google.colab import files
files.download('models_v1.zip')
